# Yelp Academic Dataset - Exploratory Analysis

Tasks 1–3: NYC Restaurant Corpus · Data Quality Audit · Temporal Coverage

**File loading strategy:** All five NDJSON files (~9 GB uncompressed) are read with
`pandas.read_json(..., lines=True, chunksize=50_000)`. The NYC restaurant `business_id` set
(Task 1) is materialised first and used to filter every downstream file in-stream - no file
is ever fully loaded into memory.

| # | Task | Phase | Priority |
|---|---|---|---|
| 1 | NYC Restaurant Corpus Definition | Foundation | P0 |
| 2 | Data Quality Audit | Foundation | P0 |
| 3 | Temporal Coverage & TLC Alignment | Foundation | P0 |

In [6]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

RAW = Path("../data/raw/yelp")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

BUSINESS_FILE = RAW / "yelp_academic_dataset_business.json"
REVIEW_FILE = RAW / "yelp_academic_dataset_review.json"
USER_FILE = RAW / "yelp_academic_dataset_user.json"
CHECKIN_FILE = RAW / "yelp_academic_dataset_checkin.json"
TIP_FILE = RAW / "yelp_academic_dataset_tip.json"

CHUNK = 50_000  # rows per chunk for all NDJSON reads

# consistent visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

## Task 1 - NYC Restaurant Corpus Definition

Filter `business.json` to businesses in New York City with "Restaurants" in their `categories`.
This corpus drives **all downstream filtering**: no other file is loaded without first
intersecting against the resulting `business_id` set.

In [8]:
import subprocess, io
from collections import Counter

NYC_CITIES = {"new york", "brooklyn", "queens", "bronx", "staten island", "new york city"}
CITY_TO_BOROUGH = {
    "new york": "Manhattan",
    "new york city": "Manhattan",
    "brooklyn": "Brooklyn",
    "queens": "Queens",
    "bronx": "Bronx",
    "staten island": "Staten Island",
}

# grep is orders of magnitude faster than pandas chunking for a targeted state filter
# on a large NDJSON file — extracts only matching lines before any parsing.
result = subprocess.run(
    ["grep", "-F", '"state":"NY"', str(BUSINESS_FILE)],
    capture_output=True, text=True
)

if not result.stdout.strip():
    print('No records with "state":"NY" found in the business file.')
    print("Scanning for available states ...\n")
    state_result = subprocess.run(
        ["grep", "-oE", '"state":"[A-Z]+"', str(BUSINESS_FILE)],
        capture_output=True, text=True
    )
    state_counts = Counter(state_result.stdout.strip().split("\n"))
    print(f"{'state':25s}  {'count':>8}")
    print("-" * 36)
    for state, count in sorted(state_counts.items(), key=lambda x: -x[1]):
        print(f"  {state:23s}  {count:8,}")
    raise RuntimeError(
        "Yelp dataset contains no NY state businesses — NYC restaurant corpus cannot be built. "
        "Pivot to one of the states listed above."
    )

# Parse matched lines directly — no full-file load needed
ny_biz = pd.read_json(io.StringIO(result.stdout), lines=True)
print(f"NY state businesses found: {len(ny_biz):,}")

# Filter to NYC city names and Restaurants category
ny_biz = ny_biz[ny_biz["city"].str.lower().isin(NYC_CITIES)]
mask = ny_biz["categories"].fillna("").str.contains("Restaurants", case=True, regex=False)
nyc_biz = ny_biz[mask].reset_index(drop=True)
print(f"NYC restaurants: {len(nyc_biz):,}")

No records with "state":"NY" found in the business file.
Scanning for available states ...

state                         count
------------------------------------
  "state":"PA"               34,039
  "state":"FL"               26,330
  "state":"TN"               12,056
  "state":"IN"               11,247
  "state":"MO"               10,913
  "state":"LA"                9,924
  "state":"AZ"                9,912
  "state":"NJ"                8,536
  "state":"NV"                7,715
  "state":"AB"                5,573
  "state":"CA"                5,203
  "state":"ID"                4,467
  "state":"DE"                2,265
  "state":"IL"                2,145
  "state":"TX"                    4
  "state":"CO"                    3
  "state":"WA"                    2
  "state":"HI"                    2
  "state":"MA"                    2
  "state":"NC"                    1
  "state":"UT"                    1
  "state":"MT"                    1
  "state":"MI"                    1
  "stat

RuntimeError: Yelp dataset contains no NY state businesses — NYC restaurant corpus cannot be built. Pivot to one of the states listed above.